# Fine-Tuning multilingual-e5-small on Databricks (CPU)

## Overview

This notebook walks through fine-tuning **multilingual-e5-small** on your own domain data using **CPU-only** compute on GCP Databricks. Fine-tuning adapts the base model's embeddings to better capture the semantic relationships specific to your data — improving retrieval, classification, and clustering quality for your domain.

### Why CPU?
At ~134M parameters (~470MB), multilingual-e5-small is small enough to fine-tune on CPU. Training will be slower than GPU (expect ~5-10x longer per epoch), but entirely viable for this model size. This is useful when GPU instances are unavailable or when you want to avoid GPU costs for smaller training runs.

> **GPU upgrade path**: If you later get GPU access, change `device="cpu"` to `device="cuda"` and optionally set `use_amp=True` in the training arguments. Everything else stays the same.

### Fine-Tuning Process
1. **Create Cluster** (GCP CPU instance, no GPU required)
2. **Install Dependencies** (transformers, sentence-transformers, datasets)
3. **Configure Storage** (Unity Catalog schema and volume)
4. **Load Training Data** (from Delta table or sample data)
5. **Fine-Tune on CPU** (CosineSimilarityLoss with gradient accumulation)
6. **Register Model** (MLflow with `llm/v1/embeddings` task type)
7. **Test** (compare before/after embeddings)

### Time Estimate
- Setup: 5-10 minutes
- Training (1,000 pairs, 3 epochs, CPU): ~15-30 minutes
- Training (10,000 pairs, 3 epochs, CPU): ~2-4 hours
- Registration: 5 minutes

### Input Prefixing Convention
multilingual-e5-small expects prefixed inputs. **Maintain this convention in your training data:**
- **Queries**: `"query: What is Databricks?"`
- **Passages**: `"passage: Databricks is a unified analytics platform..."`

## Step 1: Create Cluster

**Before running this notebook, attach it to a CPU cluster on GCP.**

### Cluster Configuration

1. Navigate to **Compute** in the Databricks left sidebar
2. Click **Create Cluster**
3. Configure with these settings:

   - **Cluster Name**: `E5 Fine-Tune CPU` (or any name you prefer)
   - **Access Mode**: `Single User`
   - **Databricks Runtime**: `15.4 LTS ML` or later (ML runtime required)
   - **Node Type**: `n2-standard-16` (16 vCPUs, 64GB RAM) or `n2-standard-32` (32 vCPUs, 128GB RAM) for larger datasets
   - **Workers**: 0 (single node is sufficient)
   - **Terminate After**: 120 minutes of inactivity (training runs can be long)

4. **Important**: You must use an ML runtime. Standard runtimes do not include MLflow and other ML libraries.

### Choosing Instance Size
- **n2-standard-16** (64GB RAM): Good for datasets up to ~50K training pairs
- **n2-standard-32** (128GB RAM): Better for larger datasets or if you want faster training via more CPU parallelism

### After Cluster Creation
1. Wait for cluster to start (typically 3-5 minutes)
2. Attach this notebook to the cluster
3. Proceed to Step 2 below

## Step 2: Install Dependencies

Installing the required Python libraries. This cell will restart the Python kernel after installation.

In [ ]:
%pip install transformers sentence-transformers datasets
%restart_python

## Step 3: Configuration

**Update the variables below with your environment details.**

In [ ]:
# ============================================================================
# CONFIGURATION - UPDATE THESE VALUES FOR YOUR ENVIRONMENT
# ============================================================================

# Unity Catalog Configuration
CATALOG_NAME = "YOUR_CATALOG_NAME"  # e.g., "shared", "main", "ml"
SCHEMA_NAME = "YOUR_SCHEMA_NAME"    # e.g., "embeddings", "models"
VOLUME_NAME = "multilingual_e5"     # Storage volume name

# Training Data Configuration
# Option A: Load from a Delta table (set this to your table name)
TRAINING_TABLE = None  # e.g., "my_catalog.my_schema.training_pairs"

# Option B: Use sample data (set TRAINING_TABLE = None to use sample data)

# Create catalog, schema, and volume if they don't exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.{VOLUME_NAME}")

# Model Configuration
BASE_MODEL_NAME = "intfloat/multilingual-e5-small"

# ============================================================================
# DERIVED PATHS - DO NOT EDIT
# ============================================================================
import os

VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}"
FINETUNED_MODEL_PATH = os.path.join(VOLUME_PATH, "models", "multilingual-e5-small-finetuned")
REGISTERED_MODEL_NAME = f"{CATALOG_NAME}.{SCHEMA_NAME}.multilingual-e5-small-finetuned"

print("Configuration Summary:")
print(f"  Catalog: {CATALOG_NAME}")
print(f"  Schema: {SCHEMA_NAME}")
print(f"  Volume Path: {VOLUME_PATH}")
print(f"  Fine-tuned Model Path: {FINETUNED_MODEL_PATH}")
print(f"  Registered Model: {REGISTERED_MODEL_NAME}")
print(f"  Training Table: {TRAINING_TABLE or 'Using sample data'}")
print(f"\nVerify the paths above are correct before proceeding.")

## Step 4: Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import mlflow
import torch
import transformers
import sentence_transformers
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from torch.utils.data import DataLoader

print(f"Transformers version: {transformers.__version__}")
print(f"Sentence Transformers version: {sentence_transformers.__version__}")
print(f"MLflow version: {mlflow.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: CPU (GPU available: {torch.cuda.is_available()})")

## Step 5: Load Training Data

Fine-tuning requires **pairs** of texts with similarity scores. Each training example needs:
- `text_a`: First text (e.g., a query)
- `text_b`: Second text (e.g., a relevant passage)
- `score`: Similarity score between 0.0 (unrelated) and 1.0 (semantically identical)

### Data Format
Your Delta table should have columns: `text_a`, `text_b`, `score`

**Remember to include the `query:` and `passage:` prefixes** in your training data — this matches the base model's expected format and produces the best results.

| text_a | text_b | score |
|--------|--------|-------|
| query: What are earnings estimates? | passage: Earnings estimates are analyst predictions of a company's future quarterly or annual earnings. | 1.0 |
| query: What are earnings estimates? | passage: The stock market opened higher on Tuesday. | 0.0 |

In [ ]:
if TRAINING_TABLE:
    # ================================================================
    # Option A: Load from your Delta table
    # ================================================================
    print(f"Loading training data from {TRAINING_TABLE}...")
    df = spark.table(TRAINING_TABLE).toPandas()
    print(f"  Loaded {len(df)} rows")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Sample:")
    print(df.head(3).to_string(index=False))

else:
    # ================================================================
    # Option B: Sample data for testing the pipeline end-to-end
    # Replace this with your own data for production fine-tuning
    # ================================================================
    print("Using sample financial news data for demonstration...\n")

    sample_data = [
        # Positive pairs (score=1.0) — query matches passage
        ("query: What are earnings estimates?",
         "passage: Earnings estimates are analyst predictions of a company's future quarterly or annual earnings per share.",
         1.0),
        ("query: What is an IPO?",
         "passage: An initial public offering (IPO) is the process by which a private company offers shares to the public for the first time.",
         1.0),
        ("query: How does inflation affect stocks?",
         "passage: Rising inflation typically pressures stock valuations by increasing input costs and prompting central banks to raise interest rates.",
         1.0),
        ("query: What is a stock split?",
         "passage: A stock split increases the number of shares outstanding while proportionally reducing the price per share, making shares more accessible to retail investors.",
         1.0),
        ("query: What is market capitalization?",
         "passage: Market capitalization is calculated by multiplying a company's share price by its total number of outstanding shares.",
         1.0),
        ("query: What is a dividend yield?",
         "passage: Dividend yield is the annual dividend payment divided by the stock price, expressed as a percentage.",
         1.0),
        ("query: What causes a bear market?",
         "passage: Bear markets are typically triggered by economic recessions, rising interest rates, geopolitical crises, or loss of investor confidence.",
         1.0),
        ("query: What is short selling?",
         "passage: Short selling involves borrowing shares and selling them with the expectation of buying them back at a lower price to profit from a decline.",
         1.0),

        # Negative pairs (score=0.0) — query does NOT match passage
        ("query: What are earnings estimates?",
         "passage: The Federal Reserve held interest rates steady at its latest meeting.",
         0.0),
        ("query: What is an IPO?",
         "passage: Crude oil prices rose 3% on supply concerns in the Middle East.",
         0.0),
        ("query: How does inflation affect stocks?",
         "passage: Tesla delivered 422,000 vehicles in Q1 2025, beating analyst expectations.",
         0.0),
        ("query: What is a stock split?",
         "passage: The unemployment rate fell to 3.8% in March according to the Bureau of Labor Statistics.",
         0.0),
        ("query: What is market capitalization?",
         "passage: Gold prices hit a new all-time high amid geopolitical uncertainty.",
         0.0),
        ("query: What is a dividend yield?",
         "passage: Apple announced a new MacBook Pro with M4 chip at WWDC.",
         0.0),
        ("query: What causes a bear market?",
         "passage: The company reported record revenue of $12.3 billion for the fiscal year.",
         0.0),
        ("query: What is short selling?",
         "passage: Congress passed a new infrastructure bill allocating $500 billion for transportation.",
         0.0),
    ]

    df = pd.DataFrame(sample_data, columns=["text_a", "text_b", "score"])
    print(f"  Created {len(df)} sample training pairs")
    print(f"  Positive pairs: {len(df[df['score'] > 0.5])}")
    print(f"  Negative pairs: {len(df[df['score'] <= 0.5])}")

print(f"\nSample rows:")
print(df.head(3).to_string(index=False))

### Convert to InputExample Format

SentenceTransformers expects training data as `InputExample` objects.

In [ ]:
train_examples = [
    InputExample(texts=[row["text_a"], row["text_b"]], label=float(row["score"]))
    for _, row in df.iterrows()
]

print(f"Created {len(train_examples)} training examples")
print(f"\nExample:")
print(f"  Text A: {train_examples[0].texts[0]}")
print(f"  Text B: {train_examples[0].texts[1]}")
print(f"  Score:  {train_examples[0].label}")

## Step 6: Fine-Tune on CPU

### Training Configuration

- **Loss function**: `CosineSimilarityLoss` — trains the model so that similar pairs have cosine similarity close to 1.0 and dissimilar pairs close to 0.0
- **Batch size**: 16 (kept small for CPU memory)
- **Gradient accumulation**: 4 steps — effectively simulates a batch size of 64 without requiring 4x the memory
- **Mixed precision**: Disabled (`use_amp=False`) since we're on CPU
- **Epochs**: 3 (increase for larger datasets, decrease if overfitting on small datasets)

### CPU Training Notes
- Training on CPU is ~5-10x slower than GPU but entirely viable for this model size
- The `n2-standard-16` instance provides 16 vCPUs — PyTorch will automatically parallelize across them
- For the sample data (16 pairs), training completes in under a minute
- For production datasets (1K-10K pairs), expect 15 minutes to several hours depending on dataset size

In [ ]:
# Load the base model on CPU
print("Loading base model on CPU...")
model = SentenceTransformer(BASE_MODEL_NAME, device="cpu")
print(f"  Model loaded on: {model.device}")
print(f"  Embedding dimension: {model.get_sentence_embedding_dimension()}")
print(f"  Max sequence length: {model.max_seq_length}")

In [ ]:
# Save baseline embeddings for comparison after training
comparison_texts = [
    "query: What are earnings estimates?",
    "passage: Earnings estimates are analyst predictions of a company's future quarterly or annual earnings per share.",
    "passage: The Federal Reserve held interest rates steady at its latest meeting.",
]

print("Generating baseline embeddings (before fine-tuning)...")
baseline_embeddings = model.encode(comparison_texts)

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"\nBaseline similarity scores:")
print(f"  Query vs relevant passage:   {cosine_similarity(baseline_embeddings[0], baseline_embeddings[1]):.4f}")
print(f"  Query vs irrelevant passage: {cosine_similarity(baseline_embeddings[0], baseline_embeddings[2]):.4f}")

In [ ]:
from sentence_transformers import SentenceTransformerTrainingArguments, SentenceTransformerTrainer
from sentence_transformers.training_args import BatchSamplers
from datasets import Dataset

# Convert training examples to HuggingFace Dataset format
train_dataset = Dataset.from_dict({
    "sentence1": [ex.texts[0] for ex in train_examples],
    "sentence2": [ex.texts[1] for ex in train_examples],
    "score": [ex.label for ex in train_examples],
})

# Configure loss function
loss = losses.CosineSimilarityLoss(model)

# ============================================================================
# TRAINING ARGUMENTS
# ============================================================================
# Adjust these based on your dataset size and available time.
#
# gradient_accumulation_steps=4 with per_device_train_batch_size=16
# effectively trains with batch size 64 while only using memory for 16.
#
# To switch to GPU later:
#   1. Change device="cpu" to device="cuda" when loading the model above
#   2. Set use_amp=True below (fp16="auto" also works)
#   3. Optionally increase per_device_train_batch_size to 32 or 64
# ============================================================================
training_args = SentenceTransformerTrainingArguments(
    output_dir=os.path.join(VOLUME_PATH, "training_output"),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    use_cpu=True,
    fp16=False,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    logging_steps=10,
    save_strategy="epoch",
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    loss=loss,
)

print("Starting fine-tuning on CPU...")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation steps: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Training examples: {len(train_dataset)}")
print()

trainer.train()

print("\nFine-tuning complete!")

In [ ]:
# Save the fine-tuned model to Volumes
print(f"Saving fine-tuned model to {FINETUNED_MODEL_PATH}...")
model.save(FINETUNED_MODEL_PATH)
print("  Saved.")

## Step 7: Register with MLflow

Register the fine-tuned model in Unity Catalog so it can be deployed as a serving endpoint.

In [ ]:
print("Registering fine-tuned model in MLflow...")

mlflow.set_registry_uri("databricks-uc")

# Set experiment in your user directory
user_name = spark.sql("SELECT current_user()").collect()[0][0]
mlflow.set_experiment(f"/Users/{user_name}/multilingual-e5-small-finetune")

input_example = ["query: What are earnings estimates?", "passage: Earnings estimates are analyst predictions."]

with mlflow.start_run():
    # Log training parameters for reproducibility
    mlflow.log_params({
        "base_model": BASE_MODEL_NAME,
        "epochs": 3,
        "batch_size": 16,
        "gradient_accumulation_steps": 4,
        "learning_rate": 2e-5,
        "training_examples": len(train_examples),
        "device": "cpu",
    })

    mlflow.sentence_transformers.log_model(
        model=model,
        artifact_path="model",
        task="llm/v1/embeddings",
        registered_model_name=REGISTERED_MODEL_NAME,
        input_example=input_example,
    )

print(f"\nModel registered successfully!")
print(f"  Registered as: {REGISTERED_MODEL_NAME}")
print(f"  Task type: llm/v1/embeddings")

## Step 8: Test Fine-Tuned Model

Compare embeddings before and after fine-tuning. After training on domain-specific pairs, the model should produce:
- **Higher** similarity for relevant query-passage pairs
- **Lower** similarity for irrelevant query-passage pairs

The gap between relevant and irrelevant similarity scores should widen.

In [ ]:
print("Generating fine-tuned embeddings (after fine-tuning)...")
finetuned_embeddings = model.encode(comparison_texts)

print(f"\n{'='*60}")
print("BEFORE vs AFTER FINE-TUNING")
print(f"{'='*60}")
print(f"\nComparison texts:")
for i, t in enumerate(comparison_texts):
    print(f"  [{i}] {t}")

baseline_relevant = cosine_similarity(baseline_embeddings[0], baseline_embeddings[1])
baseline_irrelevant = cosine_similarity(baseline_embeddings[0], baseline_embeddings[2])
finetuned_relevant = cosine_similarity(finetuned_embeddings[0], finetuned_embeddings[1])
finetuned_irrelevant = cosine_similarity(finetuned_embeddings[0], finetuned_embeddings[2])

print(f"\n{'Metric':<35} {'Before':>10} {'After':>10} {'Delta':>10}")
print(f"{'-'*65}")
print(f"{'Query vs relevant passage':<35} {baseline_relevant:>10.4f} {finetuned_relevant:>10.4f} {finetuned_relevant - baseline_relevant:>+10.4f}")
print(f"{'Query vs irrelevant passage':<35} {baseline_irrelevant:>10.4f} {finetuned_irrelevant:>10.4f} {finetuned_irrelevant - baseline_irrelevant:>+10.4f}")
print(f"{'Gap (relevant - irrelevant)':<35} {baseline_relevant - baseline_irrelevant:>10.4f} {finetuned_relevant - finetuned_irrelevant:>10.4f} {(finetuned_relevant - finetuned_irrelevant) - (baseline_relevant - baseline_irrelevant):>+10.4f}")

print(f"\nNote: With only {len(train_examples)} sample training pairs, improvements may be small.")
print(f"With your production data (1K+ pairs), expect more significant changes.")

## Step 9: Next Steps

### Deploy as a Serving Endpoint

Your fine-tuned model is now registered in Unity Catalog and ready for deployment. Follow the serving endpoint creation instructions in **[deploy_multilingual_e5_small.ipynb](deploy_multilingual_e5_small.ipynb)** (Step 6), but select your fine-tuned model (`multilingual-e5-small-finetuned`) instead of the base model.

### Improving Results

- **More data**: Fine-tuning quality scales with training data. Aim for 1,000+ pairs minimum for meaningful improvement.
- **Hard negatives**: Include challenging negative pairs (passages that look related but aren't) — these teach the model the most.
- **Balanced scores**: Include pairs across the 0.0-1.0 range, not just 0 and 1.
- **Evaluation set**: Split ~10% of your data for evaluation to monitor training progress and detect overfitting.
- **More epochs**: Try 5-10 epochs for smaller datasets (< 1K pairs), 1-3 for larger ones.

### Switching to GPU

When GPU instances become available, the only changes needed are:

```python
# Step 6: Change device
model = SentenceTransformer(BASE_MODEL_NAME, device="cuda")

# Training args: Enable mixed precision, increase batch size
training_args = SentenceTransformerTrainingArguments(
    ...
    use_cpu=False,  # remove or set False
    fp16=True,      # enable mixed precision
    per_device_train_batch_size=64,  # larger batches with GPU memory
)
```

## Fine-Tuning Summary

In [ ]:
print("=" * 60)
print("MULTILINGUAL-E5-SMALL FINE-TUNING SUMMARY")
print("=" * 60)
print(f"\nConfiguration:")
print(f"  Catalog: {CATALOG_NAME}")
print(f"  Schema: {SCHEMA_NAME}")
print(f"  Volume: {VOLUME_NAME}")
print(f"\nModel Information:")
print(f"  Base Model: {BASE_MODEL_NAME}")
print(f"  Registered Name: {REGISTERED_MODEL_NAME}")
print(f"  Task Type: llm/v1/embeddings")
print(f"  Embedding Dimensions: 384")
print(f"\nTraining Details:")
print(f"  Device: CPU")
print(f"  Training Examples: {len(train_examples)}")
print(f"  Epochs: 3")
print(f"  Effective Batch Size: 64 (16 x 4 accumulation steps)")
print(f"\nStorage:")
print(f"  Fine-tuned Model: {FINETUNED_MODEL_PATH}")
print(f"\nNext Steps:")
print(f"  1. See deploy_multilingual_e5_small.ipynb Step 6 for serving endpoint creation")
print(f"  2. Select model: {REGISTERED_MODEL_NAME}")
print(f"  3. Create Serverless endpoint (recommended)")
print("=" * 60)